# Experiment 10 - Information Retrieval

This notebook covers:
- Building a Whoosh search index
- TF-IDF and BM25F retrieval models
- Query expansion
- Evaluating IR systems (Precision, Recall, mAP)

## 1.2.1 Installation

In [1]:
!pip install whoosh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 5.5 MB/s eta 0:00:00


## 1.2.2 Preparing the Data (Stack Overflow)

> **Note:** You need a Kaggle account. Upload your `kaggle.json` API token first, or manually download the dataset.

In [2]:
# Upload kaggle.json token
from google.colab import files
import os

# Uncomment the next two lines if you need to upload your kaggle.json
# uploaded = files.upload()  # upload kaggle.json
# os.makedirs('/root/.kaggle', exist_ok=True)
# os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 600)

!pip install kaggle -q
!kaggle datasets download -d stackoverflow/stacksample
!unzip -q stacksample.zip

Dataset URL: https://www.kaggle.com/datasets/stackoverflow/stacksample
License(s): other
100% 1.11G/1.11G [00:09<00:00, 132MB/s]



In [3]:
import pandas as pd

questions = pd.read_csv('Questions.csv', nrows=20000, encoding='latin-1')
questions.head()

,Id,OwnerUserId,CreationDate,ClosedDate,Score,Title,Body
0,80,26.0,2008-08-01T13:57:07Z,NaN,26,SQLStatement.execute() - multiple queries in o...,<p>I've written a database generation script i...
1,90,58.0,2008-08-01T14:41:24Z,2012-12-26T03:45:49Z,144,Good branching and merging tutorials for Torto...,<p>Are there any really good tutorials explain...
2,120,83.0,2008-08-01T15:50:08Z,NaN,21,ASP.NET Site Maps,<p>Has anyone got experience creating <strong>...
3,180,2089740.0,2008-08-01T18:42:19Z,NaN,53,Function for creating color wheels,<p>This is something I've pseudo-solved many t...
4,260,91.0,2008-08-01T23:22:08Z,NaN,49,Adding scripting functionality to .NET applica...,<p>I have a little game written in C#. It uses...


## 1.2.3 Creating the Index and Schema

In [4]:
from whoosh.fields import Schema, TEXT, ID

# Define the index schema
schema = Schema(
    Id=ID(stored=True),
    Title=TEXT(stored=True),
    Body=TEXT(stored=True)
)

print('Schema defined:', schema.names())

Schema defined: ['Body', 'Id', 'Title']


In [5]:
import os
from whoosh.index import create_in

index_dir = 'indexdir'
if not os.path.exists(index_dir):
    os.mkdir(index_dir)

# Create the index
ix = create_in(index_dir, schema)

# Add documents
writer = ix.writer()
for idx, row in questions.iterrows():
    writer.add_document(
        Id=str(row['Id']),
        Title=str(row['Title']),
        Body=str(row['Body'])
    )
writer.commit()

print(f'Index created with {ix.doc_count()} documents.')

Index created with 20000 documents.


## 1.2.4 Searching the Index

### TF-IDF Retrieval

In [6]:
from whoosh.qparser import QueryParser
from whoosh import scoring

# Create query parser on the Title field
qp = QueryParser('Title', schema=schema)

query_sentence = 'How to install'
query = qp.parse(query_sentence)

# Search with TF-IDF
searcher_tfidf = ix.searcher(weighting=scoring.TF_IDF())
results_tfidf = searcher_tfidf.search(query, limit=3, scored=True)

print(f'Query: "{query_sentence}"')
print(f'Total matched: {len(results_tfidf)}\n')
for hit in results_tfidf:
    print(f'ID: {hit["Id"]}')
    print(f'Title: {hit["Title"]}')
    print('-' * 50)

Query: "How to install"
Total matched: 21

ID: 102850
Title: How can I install CPAN modules locally without root access (DynaLoader.pm line 229 error)?
--------------------------------------------------
ID: 145900
Title: How can I determine that Windows Installer is performing an upgrade rather than a first time install?
--------------------------------------------------
ID: 351640
Title: How to install Hibernate Tools in Eclipse?
--------------------------------------------------


### Task 1 — Test with different queries

In [7]:
test_queries = [
    'Python error',
    'SQL database connection',
    'machine learning model'
]

for q_text in test_queries:
    query = qp.parse(q_text)
    results = searcher_tfidf.search(query, limit=None)  # limit=None returns all matches
    print(f'Query: "{q_text}" => {len(results)} result(s) matched')

Query: "Python error" => 3 result(s) matched
Query: "SQL database connection" => 3 result(s) matched
Query: "machine learning model" => 0 result(s) matched


### Task 2 — BM25F Retrieval (Probabilistic Model)

In [8]:
query_sentence = 'How to install'
query = qp.parse(query_sentence)

# Search with BM25F
searcher_bm25 = ix.searcher(weighting=scoring.BM25F())
results_bm25 = searcher_bm25.search(query, limit=3, scored=True)

print(f'BM25F Results for "{query_sentence}":\n')
for hit in results_bm25:
    print(f'ID: {hit["Id"]}')
    print(f'Title: {hit["Title"]}')
    print('-' * 50)

print('\n--- Comparison ---')
print('TF-IDF top result ID:', results_tfidf[0]['Id'] if len(results_tfidf) > 0 else 'None')
print('BM25F  top result ID:', results_bm25[0]['Id']  if len(results_bm25)  > 0 else 'None')

BM25F Results for "How to install":

ID: 921780
Title: How to install ImageMagick on MAMP?
--------------------------------------------------
ID: 998260
Title: How do you install JDK?
--------------------------------------------------
ID: 351640
Title: How to install Hibernate Tools in Eclipse?
--------------------------------------------------

--- Comparison ---
TF-IDF top result ID: 102850
BM25F  top result ID: 921780


## 1.2.5 Query Expansion

In [9]:
# Re-run TF-IDF search to make sure results_tfidf is fresh
query = qp.parse('How to install')
searcher_tfidf = ix.searcher(weighting=scoring.TF_IDF())
results_tfidf = searcher_tfidf.search(query, limit=3, scored=True)

# More-like-this: find documents similar to the top result
print('Documents similar to top result:\n')
more_results = results_tfidf[0].more_like_this('Title')
for hit in more_results:
    print(f'ID: {hit["Id"]}')
    print(f'Title: {hit["Title"]}')
    print('-' * 50)

Documents similar to top result:

ID: 459590
Title: What is the difference betwen including modules and embedding modules?
--------------------------------------------------
ID: 423330
Title: Why can't DynaLoader.pm load SSleay.dll for Net::SSLeay and Crypt::SSLeay?
--------------------------------------------------
ID: 540640
Title: How can I install a CPAN module into a local directory?
--------------------------------------------------
ID: 172040
Title: How do you develop against OpenID locally
--------------------------------------------------
ID: 566290
Title: Silverlight Development - Service URL while developing locally
--------------------------------------------------
ID: 766830
Title: How can I locally manage C manuals?
--------------------------------------------------
ID: 799860
Title: Using Mercurial locally, only with Subversion server
--------------------------------------------------
ID: 852280
Title: Ubuntu: "Could not find rails locally or in a repository"
-----------

In [10]:
# Extract key terms from the top 10 results
keywords = [
    keyword
    for keyword, score in results_tfidf.key_terms('Title', docs=10, numterms=5)
]
print('Key terms extracted from top 10 results:')
print(keywords)

Key terms extracted from top 10 results:
['install', '229', 'cpan', 'dynaloader.pm', 'locally']


## 1.2.6 Evaluating IR Systems

### Toy Dataset Setup

In [11]:
queries = {
    'q1': 'machine learning',
    'q2': 'AI algorithms'
}

relevance = {
    'q1': ['doc1', 'doc2', 'doc3'],
    'q2': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']
}

documents = {
    'doc1': (
        'Artificial Intelligence (AI) is transforming various industries through automation and advanced algorithms. '
        'Machine learning, a subset of AI, enables computers to learn from data and make predictions. '
        'Algorithms are at the core of AI systems, guiding decision-making and problem-solving processes. '
        'AI-powered systems are increasingly used in healthcare for diagnosis and treatment planning. '
        'The ethical implications of AI algorithms, such as bias and fairness, are important considerations in their development.'
    ),
    'doc2': (
        'Deep learning, a branch of machine learning, uses neural networks to process complex data. '
        'AI algorithms are capable of analyzing large datasets to extract meaningful insights. '
        'Natural Language Processing (NLP) algorithms enable computers to understand and generate human language. '
        'AI-driven recommendation algorithms personalize user experiences in e-commerce and content platforms. '
        'Ensuring the transparency and accountability of AI algorithms is essential for building trust in AI technologies.'
    ),
    'doc3': (
        'Reinforcement learning algorithms enable AI agents to learn through trial and error interactions with their environment. '
        'AI algorithms are used in financial markets for high-frequency trading and risk management. '
        'Computer vision algorithms enable machines to interpret and analyze visual information. '
        'AI algorithms can enhance cybersecurity by detecting and mitigating cyber threats in real-time. '
        'Continuous research and development are essential for advancing AI algorithms and overcoming their limitations.'
    ),
    'doc4': (
        'Evolutionary algorithms, inspired by natural selection, are used to optimize complex systems and processes. '
        'AI algorithms play a crucial role in autonomous vehicles for navigation and decision-making. '
        'Quantum computing algorithms have the potential to revolutionize AI by solving complex problems exponentially faster. '
        'AI algorithms are employed in predictive maintenance to anticipate equipment failures and reduce downtime. '
        'Ethical guidelines and regulations are needed to govern the development and deployment of AI algorithms.'
    ),
    'doc5': (
        'Genetic algorithms are used to evolve solutions to optimization and search problems inspired by natural selection. '
        'AI algorithms enable personalized content recommendations in streaming services and social media platforms. '
        'Swarm intelligence algorithms mimic the collective behavior of social insects to solve optimization problems. '
        'AI algorithms are used in drug discovery to accelerate the identification of potential treatments. '
        'Collaborative efforts are essential for advancing AI algorithms and harnessing their full potential for societal benefit.'
    )
}

print('Dataset loaded:', list(documents.keys()))

Dataset loaded: ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']


In [12]:
from whoosh.fields import Schema, TEXT, ID
from whoosh.index import create_in
import os

# Define schema and create index for toy dataset
schema_toy = Schema(Id=ID(stored=True), Body=TEXT(stored=True))

index_dir_toy = 'indexdir_toy'
if not os.path.exists(index_dir_toy):
    os.mkdir(index_dir_toy)

ix_toy = create_in(index_dir_toy, schema_toy)
writer = ix_toy.writer()
for doc_id, text in documents.items():
    writer.add_document(Id=doc_id, Body=text)
writer.commit()

print(f'Toy index created with {ix_toy.doc_count()} documents.')

Toy index created with 5 documents.


In [13]:
from whoosh.qparser import QueryParser
from whoosh import scoring

qp_toy = QueryParser('Body', schema=schema_toy)

# Search for q1
query = qp_toy.parse(queries['q1'])
searcher_toy = ix_toy.searcher(weighting=scoring.TF_IDF())
results_q1 = searcher_toy.search(query, limit=3, scored=True)

print(f'Results for q1: "{queries["q1"]}"\n')
for hit in results_q1:
    print(f'  {hit["Id"]}')
print()

Results for q1: "machine learning"

  doc2
  doc1



### Task 3 — Precision and Recall for q1

In [14]:
def precision_recall(retrieved_ids, relevant_ids):
    """
    Compute Precision and Recall.

    Precision = |Retrieved ∩ Relevant| / |Retrieved|
    Recall    = |Retrieved ∩ Relevant| / |Relevant|
    """
    retrieved_set = set(retrieved_ids)
    relevant_set  = set(relevant_ids)
    tp = len(retrieved_set & relevant_set)

    precision = tp / len(retrieved_set) if retrieved_set else 0.0
    recall    = tp / len(relevant_set)  if relevant_set  else 0.0
    return precision, recall


# Get retrieved doc IDs for q1
retrieved_q1 = [hit['Id'] for hit in results_q1]
print('Retrieved:', retrieved_q1)
print('Relevant: ', relevance['q1'])

p, r = precision_recall(retrieved_q1, relevance['q1'])
print(f'\nPrecision@3 : {p:.4f}')
print(f'Recall@3    : {r:.4f}')

Retrieved: ['doc2', 'doc1']
Relevant:  ['doc1', 'doc2', 'doc3']

Precision@3 : 1.0000
Recall@3    : 0.6667


### Task 4 — Precision, Recall and mAP for all queries

In [15]:
def average_precision(retrieved_ids, relevant_ids):
    """
    Compute Average Precision (AP) — rank-aware metric.
    AP = (1/|Relevant|) * sum( Precision@k * rel(k) )
    """
    relevant_set = set(relevant_ids)
    num_relevant = len(relevant_set)
    if num_relevant == 0:
        return 0.0

    hits, ap = 0, 0.0
    for k, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            hits += 1
            ap += hits / k  # precision at position k
    return ap / num_relevant


# Evaluate all queries
print(f'{"Query":<6}  {"Retrieved":<40}  {"Precision":>10}  {"Recall":>8}  {"AP":>8}')
print('-' * 80)

ap_scores = []

for q_id, q_text in queries.items():
    query  = qp_toy.parse(q_text)
    # Re-open searcher (closed searchers raise errors on re-use)
    with ix_toy.searcher(weighting=scoring.TF_IDF()) as s:
        results = s.search(query, limit=3, scored=True)
        retrieved = [hit['Id'] for hit in results]

    p, r = precision_recall(retrieved, relevance[q_id])
    ap   = average_precision(retrieved, relevance[q_id])
    ap_scores.append(ap)

    print(f'{q_id:<6}  {str(retrieved):<40}  {p:>10.4f}  {r:>8.4f}  {ap:>8.4f}')

map_score = sum(ap_scores) / len(ap_scores)
print('-' * 80)
print(f'Mean Average Precision (mAP): {map_score:.4f}')

Query   Retrieved                                  Precision    Recall        AP
--------------------------------------------------------------------------------
q1      ['doc2', 'doc1']                              1.0000    0.6667    0.6667
q2      ['doc3', 'doc4', 'doc1']                      1.0000    0.6000    0.6000
--------------------------------------------------------------------------------
Mean Average Precision (mAP): 0.6333


### Precision@k and Recall@k at different cutoffs

In [16]:
k_values = [1, 2, 3, 5]

for q_id, q_text in queries.items():
    print(f'Query {q_id}: "{q_text}"')
    query = qp_toy.parse(q_text)
    with ix_toy.searcher(weighting=scoring.TF_IDF()) as s:
        results = s.search(query, limit=max(k_values), scored=True)
        all_retrieved = [hit['Id'] for hit in results]

    for k in k_values:
        top_k = all_retrieved[:k]
        p, r  = precision_recall(top_k, relevance[q_id])
        print(f'  P@{k}={p:.4f}  R@{k}={r:.4f}')
    print()

Query q1: "machine learning"
  P@1=1.0000  R@1=0.3333
  P@2=1.0000  R@2=0.6667
  P@3=1.0000  R@3=0.6667
  P@5=1.0000  R@5=0.6667

Query q2: "AI algorithms"
  P@1=1.0000  R@1=0.2000
  P@2=1.0000  R@2=0.4000
  P@3=1.0000  R@3=0.6000
  P@5=1.0000  R@5=1.0000

